# Diagnosing and fixing less than 80% genes mapped to TF error in pyscenic 

In [1]:
## check genes against pyscenic feather
import pandas as pd
import polars as pl
import numpy as np
import os
import loompy
from ctxcore.rnkdb import FeatherRankingDatabase as RankingDB

In [2]:
os.chdir("/rds/projects/g/gendood-3dmucosa/scRNAseqAnalysis/")

In [3]:
seurat_genes_path = "human_epi_mtx/human_epi_counts.csv"
ranking_feather_path = "pyscenic_resources/hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather"
ranking_feather_path2 = "pyscenic_resources/hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather"
adj_path = "pyscenic_res/human_epi.adjacencies.tsv"

In [4]:
ranking_feather = pd.read_feather(ranking_feather_path, columns=None, use_threads=True)
ranking_feather2 = pd.read_feather(ranking_feather_path2, columns=None, use_threads=True)

In [ ]:
## get alias_mapping file
%curl -L -o Homo_sapiens.gene_info.gz   https://ftp.ncbi.nlm.nih.gov/gene/DATA/GENE_INFO/Mammalia/Homo_sapiens.gene_info.gz
%gunzip Homo_sapiens.gene_info.gz

In [5]:
# set the correct separator: "\t" for TSV, "," for CSV
pldf = pl.read_csv(
    seurat_genes_path,
    separator=",",          # <-- change to "," if it’s truly comma CSV
    infer_schema_length=1000,
    ignore_errors=True
)

# make first column the index after converting to pandas (ensures NumPy dtypes, not Arrow/nullable)
counts_df = pldf.to_pandas(use_pyarrow_extension_array=False)
counts_df.set_index(counts_df.columns[0], inplace=True)

# ensure pure numeric dtypes (good for AUCell) and cut memory
counts_df = counts_df.apply(pd.to_numeric, errors="coerce").fillna(0).astype(np.int32)
seurat_genes = counts_df.columns

## Disagnosing the issue

In [6]:
ranking_genes=ranking_feather.columns[:-1]
ranking_genes2=ranking_feather2.columns[:-1]

In [7]:
intersect_set=set(ranking_genes).intersection(set(seurat_genes))
intersect_set2=set(ranking_genes2).intersection(set(seurat_genes))

In [8]:
ranked_only = set(ranking_genes) - set(seurat_genes)
ranked_2_only = set(ranking_genes2) - set(seurat_genes)
seurat_only = set(seurat_genes) - set(ranking_genes)

In [9]:
print(f'''
total genes in ranked = {len(ranking_genes)}
total genes in ranked_2 = {len(ranking_genes2)}
total genes in seurat = {len(seurat_genes)}
genes in seurat and 10kb window = {len(intersect_set)}
genes in seurat and 500b window = {len(intersect_set2)}
genes in ranked_only = {len(ranked_only)}
genes in ranked_2_only = {len(ranked_2_only)}
genes in seurat_only = {len(seurat_only)}
''')


total genes in ranked = 27090
total genes in ranked_2 = 27015
total genes in seurat = 58920
genes in seurat and 10kb window = 24591
genes in seurat and 500b window = 24529
genes in ranked_only = 2499
genes in ranked_2_only = 2486
genes in seurat_only = 34329



In [12]:
"TP63" in intersect_set and "RUNX1" in intersect_set

True

In [13]:
"TP63" in intersect_set2 and "RUNX1" in intersect_set2

True

In [14]:
ranking_feather.head()

,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,A3GALT2,A4GALT,A4GNT,...,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,motifs
0,26008,25991,1658,11148,21811,6185,19258,6095,12391,14038,...,7724,837,3034,9074,25479,22125,5926,10403,10303,bergman__Su_H_
1,15420,17248,1033,3379,11607,16530,21254,12537,6770,14834,...,22917,22790,21715,19846,16860,1319,12561,18506,13679,bergman__croc
2,24149,25266,4179,6054,24627,11654,14256,22012,19372,17183,...,9475,11660,17333,26034,21832,16151,6184,15945,242,bergman__pho
3,435,23027,4349,13181,18422,13483,3861,21954,6992,17098,...,20819,13403,13498,8824,10179,18403,6122,13180,4113,bergman__tll
4,24824,24940,13774,13039,25467,12261,1457,24482,10578,9066,...,17010,7624,13470,18363,18925,1740,16752,8465,2540,c2h2_zfs__M0369


In [15]:
print(list(ranked_only)[0:10], list(seurat_only)[0:10])

['LOC101927640', 'LOC101927755', 'C17orf112', 'LOC101929753', 'MLLT4', 'TMEM211', 'LOC100287728', 'GGTA1P', 'LOC101927746', 'LOC101929415'] ['RNA5SP457', 'RNU6-862P', 'SNRPGP1', 'LOC124906067', 'LOC105370041', 'NPY2R-AS1', 'GREB1L-DT', 'LOC122526776', 'LOC124903414', 'EIF4E2P2']


In [16]:
# ---- 0) Setup ---------------------------------------------------------------
import re
from ctxcore.rnkdb import FeatherRankingDatabase as RankingDB

# counts_df: your expression matrix as pandas DataFrame
#   rows=genes, cols=cells  (if your genes are columns, adapt below)
# ranking_feather: path to *.rankings.feather

# If your genes are columns (cells x genes), flip this:
GENES_ARE_INDEX = False

# db_fnames = glob.glob(DATABASES_GLOB)
# def name(fname):
#     return os.path.splitext(os.path.basename(fname))[0]
# dbs = [RankingDatabase(fname=fname, name=name(fname)) for fname in db_fnames]
# dbs

# ---- 1) Pull gene list from ranking feather (cheap; no full table read) ----
db = RankingDB(name = ranking_feather_path, fname = ranking_feather_path)
# rank_genes_raw = pd.Index(db.genes())  # returns a list of gene names in the DB

g = getattr(db, "genes", None)
rank_genes_raw = pd.Index(g() if callable(g) else g)

# # ---- 2) Get expression genes ------------------------------------------------
expr_genes_raw = counts_df.index if GENES_ARE_INDEX else counts_df.columns
expr_genes_raw = pd.Index(expr_genes_raw)

# # ---- 3) Helpers to detect/normalize ID styles -------------------------------
def looks_like_ensembl(s: pd.Index) -> bool:
    # "ENSG00000123456" with optional version ".12"
    head = s[: min(1000, len(s))].astype(str)
    m = head.str.match(r"^ENS[A-Z]{0,4}\d+(\.\d+)?$")
    # consider "looks like ensembl" if >=70% match the pattern
    return m.mean() >= 0.7

def normalize_symbols(idx: pd.Index) -> pd.Index:
    return pd.Index(idx.astype(str).str.strip().str.upper())

def strip_ensembl_version(idx: pd.Index) -> pd.Index:
    return pd.Index(idx.astype(str).str.replace(r"\.\d+$", "", regex=True))

# Detect styles
expr_is_ens = looks_like_ensembl(expr_genes_raw)
rank_is_ens = looks_like_ensembl(rank_genes_raw)

# ---- 4) Normalize into a *common* namespace (best effort) -------------------
if expr_is_ens and rank_is_ens:
    expr_genes = strip_ensembl_version(expr_genes_raw)
    rank_genes = strip_ensembl_version(rank_genes_raw)
elif (not expr_is_ens) and (not rank_is_ens):
    expr_genes = normalize_symbols(expr_genes_raw)
    rank_genes = normalize_symbols(rank_genes_raw)
elif expr_is_ens and (not rank_is_ens):
    # You will eventually need an Ensembl->Symbol map for perfect matching.
    # For now, strip versions and compare raw strings (often still 0 overlap).
    expr_genes = strip_ensembl_version(expr_genes_raw)  # still Ensembl IDs
    rank_genes = normalize_symbols(rank_genes_raw)      # symbols
else:  # expr symbols, rank ensembl
    expr_genes = normalize_symbols(expr_genes_raw)
    rank_genes = strip_ensembl_version(rank_genes_raw)

# ---- 5) Compute overlaps (like your summary) --------------------------------
in_both = expr_genes.intersection(rank_genes)
expr_only = expr_genes.difference(rank_genes)
rank_only = rank_genes.difference(expr_genes)

print(f"total genes in ranked = {len(rank_genes)}")
print(f"total genes in seurat = {len(expr_genes)}")
print(f"genes in seurat ∩ ranked = {len(in_both)}")
print(f"genes in ranked only = {len(rank_only)}")
print(f"genes in seurat only = {len(expr_only)}")

# ---- 6) Quick check for TFs of interest -------------------------------------
for tf in ["TP63", "RUNX1"]:
    tf_norm = tf.upper()
    print(
        tf,
        "in expr?", tf_norm in expr_genes.values,
        "| in ranked?", tf_norm in rank_genes.values
    )

# ---- 7) (Optional) rewrite counts_df index to normalized names --------------
# This maximizes mapping downstream. ONLY do this if your genes are indeed
# symbols (or Ensembl) consistently after detection above.
if GENES_ARE_INDEX:
    if expr_is_ens and rank_is_ens:
        counts_df.index = strip_ensembl_version(counts_df.index)
    elif (not expr_is_ens) and (not rank_is_ens):
        counts_df.index = normalize_symbols(counts_df.index)
    # If one side is Ensembl and the other is symbols, you should build a proper
    # Ensembl<->Symbol map (e.g., from a GTF) and remap counts_df.index before proceeding.


total genes in ranked = 27090
total genes in seurat = 58920
genes in seurat ∩ ranked = 24591
genes in ranked only = 2499
genes in seurat only = 34329
TP63 in expr? True | in ranked? True
RUNX1 in expr? True | in ranked? True


In [17]:
counts_pdf = None   # e.g. a pandas counts matrix
counts_pldf = counts_df  # e.g. a polars counts matrix
# If you have any non-gene columns (e.g. 'cell_id'), list them here:
NON_GENE_COLS = []  # e.g. ["cell_id"] or leave empty if columns are purely genes

# --- 1) Expression gene universe = COLUMNS (cells x genes) ---
schema_names = counts_pldf.columns
gene_cols = [c for c in schema_names if c not in NON_GENE_COLS]
expr_genes = pd.Index(gene_cols).astype(str).str.strip().str.upper()
expr_genes_set = set(expr_genes)

# --- 2) Ranking DB gene universe ---
db = RankingDB(name = ranking_feather_path, fname = ranking_feather_path)
g = getattr(db, "genes", None)
rank_genes_raw = g() if callable(g) else g
rank_genes = pd.Index(rank_genes_raw).astype(str).str.strip().str.upper()
rank_genes_set = set(rank_genes)

# --- 3) Read massive adjacencies with Polars (TSV), TF/target only ---
adj = (
    pl.read_csv(
        adj_path,
        separator="\t",
        columns=["TF", "target"],
        dtypes={"TF": pl.Utf8, "target": pl.Utf8},
        ignore_errors=True
    )
    .with_columns([
        pl.col("TF").str.strip().str.to_uppercase(),
        pl.col("target").str.strip().str.to_uppercase()
    ])
)

# Per-TF unique target sets
targets_df = adj.groupby("TF").agg(pl.col("target").unique().alias("targets"))
tf2targets = {row["TF"]: set(row["targets"]) for row in targets_df.iter_rows(named=True)}

# --- 4) Debug TP63 / RUNX1 (or any TFs) ---
def frac(a, b): return (len(a & b) / len(a)) if a else 0.0

for tf in ["TP63", "RUNX1"]:
    T = tf2targets.get(tf)
    if T is None:
        print(f"{tf}: not in adjacencies.tsv")
        continue
    f_rank = frac(T, rank_genes_set)   # mapping into ranking DB universe  (~reason #2)
    f_expr = frac(T, expr_genes_set)   # mapping into your expression columns (~reason #4)
    print(f"{tf}: targets={len(T)} | map_to_rankDB={f_rank:.1%} | map_to_expr={f_expr:.1%}")
    if f_rank < 0.80:
        miss = sorted(list(T - rank_genes_set))[:20]
        print("  missing_in_rankDB (examples):", miss)
    if f_expr < 0.80:
        miss = sorted(list(T - expr_genes_set))[:20]
        print("  missing_in_expr (examples):", miss)

# --- 5) Optional: see which TFs would be skipped (<80% in rank DB) ---
rows = []
for tf, T in tf2targets.items():
    rows.append({
        "TF": tf,
        "n_targets": len(T),
        "frac_in_rankDB": frac(T, rank_genes_set),
        "frac_in_expr":   frac(T, expr_genes_set),
    })
summary = pd.DataFrame(rows).sort_values("frac_in_rankDB")
skipped_like = summary[summary["frac_in_rankDB"] < 0.80]
print("Likely skipped by ctx (<80% in ranking DB):", skipped_like.shape[0])
print(skipped_like.head(10))

TP63: targets=8580 | map_to_rankDB=76.8% | map_to_expr=100.0%
  missing_in_rankDB (examples): ['ABITRAM', 'ACAA2P1', 'ACP3', 'ACTBP8', 'ACTE1P', 'ACTG1P14', 'ACTG1P16', 'ADGRD2', 'ADIRF-AS1', 'ADPRS', 'ADSS1', 'ADSS2', 'AFDN', 'AGGF1P1', 'AK2P1', 'AKR1C7P', 'AKT3-IT1', 'ALG1L7P', 'AMYP1', 'ANKRD65-AS1']
RUNX1: targets=12318 | map_to_rankDB=75.2% | map_to_expr=100.0%
  missing_in_rankDB (examples): ['ACAA2P1', 'ACTG1P3', 'ADAM1B', 'ADPRS', 'ADSS2', 'AFDN', 'AFF1-AS1', 'AHSA2P', 'AK2P2', 'AK3P3', 'AK4P1', 'AKAP17A', 'AKIRIN2P1', 'AKR1B10P1', 'AKR1B1P2', 'AKR1C7P', 'ALDH1A3-AS1', 'ALDOAP2', 'ALG1L12P', 'ALG1L6P']
Likely skipped by ctx (<80% in ranking DB): 1456
          TF  n_targets  frac_in_rankDB  frac_in_expr
699   ZNF423        153        0.261438           1.0
785     PAX7        157        0.363057           1.0
525     PAX3        810        0.507407           1.0
751    PRRX1       1363        0.541453           1.0
80    HSPA1L        926        0.543197           1.0
1530  ZNF

In [18]:
"MLLT4" in ranking_genes ## check for AFDN alias

True

## Creating Alias Map df

In [134]:
gi = pl.read_csv(
    "../Refs/Homo_sapiens.gene_info",   # works with .gz too
    separator="\t",
    has_header=True,
    null_values=["", "-"],
    columns=["Symbol", "Synonyms"],
)

gi = gi.select([
    pl.coalesce([pl.col("Symbol")])
      .str.to_uppercase()
      .alias("approved_symbol"),
    pl.col("Synonyms").fill_null("").str.to_uppercase().alias("Synonyms"),
])

# Explode pipe-separated synonyms -> one alias per row
alias_rows = (
    gi.filter(pl.col("Synonyms") != "")
      .with_columns(pl.col("Synonyms").str.split("|"))
      .explode("Synonyms")
      .rename({"Synonyms": "from"})
      .with_columns(pl.col("from").str.strip())
      .filter((pl.col("from") != "") & (pl.col("from") != pl.col("approved_symbol")))  # drop empties & self-maps
      .select(["from", pl.col("approved_symbol").alias("to")])
      .unique()
)

# Drop ambiguous aliases (alias mapping to multiple approved symbols)
amb = alias_rows.groupby("from").agg(pl.col("to").n_unique().alias("n_to"))
alias_map = (
    alias_rows.join(amb, on="from", how="inner")
              .filter(pl.col("n_to") == 1)
              .drop("n_to")
)

# (Optional) restrict to genes your ranking DB can score:
# alias_map = alias_map.filter(pl.col("to").is_in(rank_genes_pl))

alias_map.write_csv("hgnc_aliases.tsv", separator="\t")
alias_map.filter(pl.col("from") == "MLLT4")

from,to
str,str
"""MLLT4""","""AFDN"""


In [21]:
alias_map = pd.read_csv("hgnc_aliases.tsv", sep= "\t")
print(alias_map[alias_map["to"] == "AFDN"])
print(alias_map[alias_map["from"] == "ACAA2P1"])
print(
len(set(alias_map["to"]).intersection(set(seurat_genes))),
len(set(alias_map["from"]).intersection(set(seurat_genes))),
len(set(alias_map["to"]).intersection(set(ranking_genes))),
len(set(alias_map["from"]).intersection(set(ranking_genes2))),
len(set(alias_map["to"]).intersection(set(seurat_genes))) + len(set(alias_map["from"]).intersection(set(seurat_genes)))
)
# len(set(alias_map["to"]) - (set(seurat_genes)))
# len(set(alias_map["from"]) - (set(seurat_genes)))

           from    to
1276    MLL-AF6  AFDN
9481        AF6  AFDN
42341     MLLT4  AFDN
42342  L-AFADIN  AFDN
Empty DataFrame
Columns: [from, to]
Index: []
25674 1559 18469 2129 27233


# Replace as many gene names with aliases in counts_df as possible to be compliant with gene names in feather files and export to loom

Due to many to 1 relationship of many aliases to their HGNC symbols in the original counts_df, have decided to modify the counts_df genes to avoid accidentally losing columns in feather file

In [123]:
ranked_only = set(ranking_genes) - set(seurat_genes)
seurat_only = set(seurat_genes) - set(ranking_genes)
ranked_2_only = set(ranking_genes2) - set(seurat_genes)
alias_from_ranked =ranked_only.intersection(set(alias_map["from"]))
alias_from_ranked_2 =ranked_2_only.intersection(set(alias_map["from"]))
alias_to_ranked =ranked_only.intersection(set(alias_map["to"]))
alias_to_ranked_2 =ranked_2_only.intersection(set(alias_map["to"]))

print(f'''
length ranked_only = {len(ranked_only)}
length ranked_2_only = {len(ranked_2_only)}
genes from alias_map FROM in ranked_only = {len(alias_from_ranked)}
genes from alias_map FROM in ranked_2_only = {len(alias_from_ranked_2)}
genes from alias_map TO in ranked_only = {len(alias_to_ranked)}
genes from alias_map TO in ranked_2_only = {len(alias_to_ranked_2)}
''')

alias_from_ranked == alias_from_ranked_2 ## therefore can perform same inner join on both
## rename feather file cols based on the from col
ranking_feather_genes = pd.Series(list(ranked_only), name = "ranking_feather_genes")
ranking_feather_genes = pd.DataFrame(ranking_feather_genes)
map_df = ranking_feather_genes.merge(alias_map,left_on = "ranking_feather_genes", right_on = "from")
seurat_only_gene_aliases = pd.DataFrame(pd.Series(list(seurat_only), name = "seurat_only_gene_aliases"))
map_df = map_df.merge(seurat_only_gene_aliases, left_on = "to", right_on = "seurat_only_gene_aliases")
## gives me the complete map from the from -> to cols in the alias_map df
display(map_df)

# display(map_df[map_df["to"].isin(["PALM2AKAP2", "DGCR5"])]) ## because of these duplicate ids will have to modify counts_df col instea
## because of duplicates of the former after matching alias definitions, will just use the hyphenated version for the fusion isoform
## that can be produced
map_df = map_df[~map_df['from'].isin(['AKAP2', 'PALM2'])]

# display(map_df[map_df["to"].isin(["PALM2AKAP2", "DGCR5"])])

# map_df has columns ['from','to'] (your mapping)
rename_map = (
    map_df.drop_duplicates(subset=["to"])
          .set_index("to")["from"]
          .to_dict()
)





print(len(rename_map.values()))
print(len(set(rename_map.values())))

print(rename_map.get("PALM2AKAP2")) 

newlist = [] # empty list to hold unique elements from the list
duplist = [] # empty list to hold the duplicate elements from the list
for i in rename_map.values():
    if i not in newlist:
        newlist.append(i)
    else:
        duplist.append(i) # this method catches the first duplicate entries, and appends them to the list4

# The next step is to print the duplicate entries, and the unique entries
print("List of duplicates", duplist) 

## no duplicates can continue with the col rename to construct new count matrix to rerun

gene_col = counts_df.columns
current = set(counts_df.columns.astype(str))
safe_map = {new: old for old, new in rename_map.items()
            if (new == old) or (new not in current)}

flipped_safe_map = {v: k for k, v in safe_map.items()}
flipped_safe_map    
#     # 2) build a safe mapping to avoid duplicate index labels

    
# rename columns in-place



## now do the same with the opposite map_df to catch final 5 if poss

map_df_inverse = ranking_feather_genes.merge(alias_map,left_on = "ranking_feather_genes", right_on = "to")
map_df_inverse = map_df_inverse.merge(seurat_only_gene_aliases, left_on = "from", right_on = "seurat_only_gene_aliases")



map_df_inverse ## only 1 mismatch now

rename_map_inv_mapping = (
    map_df_inverse.drop_duplicates(subset=["to"])
          .set_index("from")["to"]
          .to_dict()
)
print(rename_map_inv_mapping)
map_df_inverse
counts_df.rename(columns=rename_map_inv_mapping, inplace=True) 


len(set(counts_df.columns) - (set(ranking_feather.columns)))

# counts_df.to_csv("human_epi_mtx/pyscenic_compatible_human_epi_counts.csv") ## takes too much time for a 1.06GB file so export to loom

import loompy
# If counts_df is cells × genes
counts_T = counts_df.T   # Now rows=genes, cols=cells

loompy.create(
    "human_epi_mtx/pyscenic_compatible_human_epi_counts.loom",
    counts_T.values,
    row_attrs={"Gene": counts_T.index.values},     # gene names
    col_attrs={"Cell": counts_T.columns.values}    # cell IDs
)
print(f'''
pyscenic_compatible loom file generated with {len(rename_map_inv_mapping) + len(flipped_safe_map)} gene alias
replacements
''')
### 34329 before
#     # (optional) if duplicates somehow exist already and you want to keep first occurrence:
#     # dup_mask = df.columns.duplicated(keep="first")
#     # if dup_mask.any():
#     #     df = df.loc[:, ~dup_mask]
    
#     df.to_feather(dst)
#     print(f"[ok] wrote {dst} | attempted renames: {len(rename_map)} | applied (safe): {len(safe_map)}")
# seurat_only = set(seurat_genes) - set(ranking_genes)
# seurat_only_2 = set(seurat_genes) - set(ranking_genes)
# seurat_genes_ls = list(seurat_genes)
# seurat_from_subset = alias_map[alias_map["from"].isin(seurat_genes_ls)]

# sg  = pd.Index(seurat_genes_ls).astype(str).str.upper()
# to  = alias_map["to"].astype(str).str.upper()
# frm = alias_map["from"].astype(str).str.upper()

# # the unique symbols you want: in `to` ∩ seurat, but NOT in `from`
# genes = sg.intersection(to).difference(frm)
# genes

# len(set(alias_map["from"]).intersection(set(seurat_genes)))
# len(set(alias_map["to"]).intersection(set(seurat_genes)))
# len(set(alias_map["to"]).intersection(set(ranking_genes)))
# len(set(alias_map["from"]).intersection(set(ranking_genes)))


length ranked_only = 2499
length ranked_2_only = 2486
genes from alias_map FROM in ranked_only = 953
genes from alias_map FROM in ranked_2_only = 953
genes from alias_map TO in ranked_only = 5
genes from alias_map TO in ranked_2_only = 5



,ranking_feather_genes,from,to,seurat_only_gene_aliases
0,MLLT4,MLLT4,AFDN,AFDN
1,TMEM211,TMEM211,LHFPL7,LHFPL7
2,ATP5J,ATP5J,ATP5PF,ATP5PF
3,MARCH2,MARCH2,MARCHF2,MARCHF2
4,WDR34,WDR34,DYNC2I2,DYNC2I2
...,...,...,...,...
903,DPCR1,DPCR1,MUCL3,MUCL3
904,KIAA1324,KIAA1324,ELAPOR1,ELAPOR1
905,LINC00684,LINC00684,FAM236A,FAM236A
906,LINC01468,LINC01468,LNCAROD,LNCAROD


906
906
PALM2-AKAP2
List of duplicates []
{'CDKN2A-DT': 'CDKN2A-AS1'}


1